# VEA Algorithm: Experiment Pipeline

This notebook demonstrates the complete extraction pipeline for processing videos using the **VEA_algo** framework.

## Install Dependencies

In [ ]:
!pip install torch==2.8.0 torchvision==0.23.0 torchaudio==2.8.0 --index-url https://download.pytorch.org/whl/cu126
!pip install torch_geometric
!pip install pyg_lib torch_scatter torch_sparse torch_cluster -f https://data.pyg.org/whl/torch-2.8.0+cu126.html
!pip install transformers==4.57.6
!pip install whisperx==3.8.4

In [ ]:
!pip install librosa
!pip install scenedetect[opencv]
!pip install decord
!pip install qwen-vl-utils
!pip install bitsandbytes
!pip install evaluate
!pip install datasets

In [ ]:
import torch
import torchvision
import torchaudio
import torch_geometric
import pyg_lib
import torch_scatter
import torch_sparse
import torch_cluster

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Torchvision Version: {torchvision.__version__}")
print(f"Torchaudio Version: {torchaudio.__version__}")
print(f"PyG version: {torch_geometric.__version__}")
print(f"pyg-lib version: {pyg_lib.__version__}")
print(f"torch-scatter version: {torch_scatter.__version__}")
print(f"torch-sparse version: {torch_sparse.__version__}")
print(f"torch-cluster version: {torch_cluster.__version__}")

In [ ]:
!pip show whisperx
!pip show transformers
!pip show librosa
!pip show scenedetect
!pip show decord
!pip show qwen-vl-utils
!pip show bitsandbytes
!pip show evaluate
!pip show datasets

## Download EnTube

In [ ]:
!python /content/drive/MyDrive/KhoaLuan/VEA/prepare_data/download_data_entube.py

## Extract Audio & Transcript & Diarization

In [ ]:
!python /content/drive/MyDrive/KhoaLuan/VEA/multimodal_to_text/extract_audio_transcribe_diarization.py \
  --root_dir /content/drive/MyDrive/KhoaLuan/EnTube/Download_2min \
  --num_workers 2 \
  --diarization_model 'community-1' \
  --whisper_model_path /content/drive/MyDrive/KhoaLuan/models/faster-whisper-large-v2 \
  --diarization_model_path /content/drive/MyDrive/KhoaLuan/models/speaker-diarization-community-1

## Process Multimodal to Text

In [ ]:
!python /content/drive/MyDrive/KhoaLuan/VEA/src/multimodal_to_text/process_video_multimodal.py \
  --root_dir /content/drive/MyDrive/KhoaLuan/EnTube/Download_2min \
  --beats_checkpoints /content/drive/MyDrive/KhoaLuan/models/beats_checkpoints/BEATs_iter3_plus_AS2M_finetuned_on_AS2M_cpt2.pt \
  --qwen_visual_model_path /content/drive/MyDrive/KhoaLuan/models/Qwen3-VL-2B-Instruct \
  --qwen_instruct_model_path /content/drive/MyDrive/KhoaLuan/models/Qwen3-4B-Instruct-2507 \
  --qwen_vl_embedding_model_path /content/drive/MyDrive/KhoaLuan/models/Qwen3-VL-Embedding-2B \
  --delete_clips

## Create RST Graph from Caption

In [ ]:
!python /content/drive/MyDrive/KhoaLuan/VEA/rst_tree_parsing/prepare_edus.py \
    --root_dir /content/drive/MyDrive/KhoaLuan/EnTube/Download_2min \
    --output_json /content/drive/MyDrive/KhoaLuan/EnTube/edu_dataset.json

In [ ]:
!python /content/drive/MyDrive/KhoaLuan/VEA/rst_tree_parsing/rst_tree_parsing.py \
    --model_size 7b \
    --corpus rstdt \
    --dataset_file /content/drive/MyDrive/KhoaLuan/EnTube/edu_dataset.json

In [ ]:
!python /content/drive/MyDrive/KhoaLuan/VEA/rst_tree_parsing/caption_rst_graph_builder.py \
    --videos-root /content/drive/MyDrive/KhoaLuan/EnTube/Download_2min \
    --label-map-json /content/drive/MyDrive/KhoaLuan/EnTube/video_ids_label.json --force

## Extract Subgraph

### Train RGCN on Dataset

In [ ]:
!python /content/drive/MyDrive/KhoaLuan/VEA/extract_subgraph/split_dataset.py \
    --data_root /content/drive/MyDrive/KhoaLuan/EnTube/Download_2min \
    --split_dataset_file /content/drive/MyDrive/KhoaLuan/EnTube/dataset_splits_test_100.json \
    --train_ratio 0.61 --val_ratio 0.075 --test_ratio 0.315

!python /content/drive/MyDrive/KhoaLuan/VEA/extract_subgraph/train_gnn.py \
    --data_root /content/drive/MyDrive/KhoaLuan/EnTube/Download_2min \
    --split_dataset_file /content/drive/MyDrive/KhoaLuan/EnTube/dataset_splits_test_100.json \
    --save_dir /content/drive/MyDrive/KhoaLuan/EnTube/models/rgcn \
    --hidden_dim 512 \
    --num_layers 3 \
    --train_batch_size 16 \
    --epochs 50 \
    --lr 1e-4 \
    --dropout 0.2 \
    --device cuda

### Extract Important Subgraph (Explaination) with SubgraphX

In [ ]:
!python /content/drive/MyDrive/KhoaLuan/VEA/extract_subgraph/run_subgraphx.py \
    --data_root /content/drive/MyDrive/KhoaLuan/EnTube/Download_2min \
    --model_path /content/drive/MyDrive/KhoaLuan/EnTube/models/rgcn/20260609_105152/best_rgcn_model.pth \
    --output_dir /content/drive/MyDrive/KhoaLuan/EnTube/SubgraphX_Results \
    --split_json /content/drive/MyDrive/KhoaLuan/EnTube/dataset_splits.json \
    --split val \
    --n_min 6 \
    --m 15 --t 10 --exp_weight 5.0 --max_children 12

!python /content/drive/MyDrive/KhoaLuan/VEA/extract_subgraph/run_subgraphx.py \
    --data_root /content/drive/MyDrive/KhoaLuan/EnTube/Download_2min \
    --model_path /content/drive/MyDrive/KhoaLuan/EnTube/models/rgcn/20260609_105152/best_rgcn_model.pth \
    --output_dir /content/drive/MyDrive/KhoaLuan/EnTube/SubgraphX_Results \
    --split_json /content/drive/MyDrive/KhoaLuan/EnTube/dataset_splits.json \
    --split train \
    --n_min 6 \
    --m 15 --t 10 --exp_weight 5.0 --max_children 12

## Build Knowledge Base

### Upload Important Subgraph (*_explaination.pt) to Neo4j

In [ ]:
!pip install neo4j

In [ ]:
%env NEO4J_URI=neo4j+s://....databases.neo4j.io
%env NEO4J_USERNAME=...
%env NEO4J_PASSWORD=...
%env NEO4J_DATABASE=...
%env AURA_INSTANCEID=...
%env AURA_INSTANCENAME=EnTubeGraph

In [ ]:
!python /content/drive/MyDrive/KhoaLuan/VEA/knowledge_graph/upload_subgraph.py \
  --data_root /content/drive/MyDrive/KhoaLuan/EnTube/Download_2min \
  --subgraph_dir /content/drive/MyDrive/KhoaLuan/EnTube/SubgraphX_Results \
  --force

### Merge Subgraph on Neo4j

In [ ]:
!python /content/drive/MyDrive/KhoaLuan/VEA/knowledge_graph/merge_subgraph.py \
  --labels 0 1 2 \
  --top_k 5 \
  --threshold 0.85

### Upload Scene Embedding to Milvus

In [ ]:
!pip install pymilvus

In [ ]:
%env MILVUS_CLUSTER_ENDPOINT=https://....serverless.aws-eu-central-1.cloud.zilliz.com
%env MILVUS_TOKEN=...
%env MILVUS_COLLECTION_NAME=video_scenes_collection

In [ ]:
!python /content/drive/MyDrive/KhoaLuan/VEA/knowledge_graph/upload_embeddings.py \
    --data_root /content/drive/MyDrive/KhoaLuan/EnTube/Download_2min \
    --subgraph_dir /content/drive/MyDrive/KhoaLuan/EnTube/SubgraphX_Results \
    --force

## Inference

In [ ]:
!pip install pymilvus neo4j

In [ ]:
%env NEO4J_URI=neo4j+s://....databases.neo4j.io
%env NEO4J_USERNAME=...
%env NEO4J_PASSWORD=...
%env NEO4J_DATABASE=...
%env AURA_INSTANCEID=...
%env AURA_INSTANCENAME=EnTubeGraph

%env MILVUS_CLUSTER_ENDPOINT=https://....serverless.aws-eu-central-1.cloud.zilliz.com
%env MILVUS_TOKEN=...
%env MILVUS_COLLECTION_NAME=video_scenes_collection

### Setup Milvus and Neo4j

In [ ]:
import os
import torch
from pymilvus import MilvusClient
from neo4j import GraphDatabase


# --- CẤU HÌNH MILVUS CLOUD ---
CLUSTER_ENDPOINT = os.getenv("MILVUS_CLUSTER_ENDPOINT")
TOKEN = os.getenv("MILVUS_TOKEN")
COLLECTION_NAME = os.getenv("MILVUS_COLLECTION_NAME")

# Khởi tạo kết nối Milvus
milvus_client = MilvusClient(uri=CLUSTER_ENDPOINT, token=TOKEN)
print("[INFO] Ket noi Milvus Cloud thanh cong.")

# --- CẤU HÌNH NEO4J AURADB ---
URI = os.getenv("NEO4J_URI")
USERNAME = os.getenv("NEO4J_USERNAME")
PASSWORD = os.getenv("NEO4J_PASSWORD")
DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")

# Khởi tạo kết nối Neo4j
driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))

try:
    driver.verify_connectivity()
    print("[INFO] Ket noi Neo4j AuraDB thanh cong! San sang cho buoc xu li tiep theo.")
except Exception as e:
    print(f"[ERROR] Khong the ket noi Neo4j. Chi tiet: {e}")

In [ ]:
import json
import torch
from pathlib import Path

# Config path
split_dataset_file = "/content/drive/MyDrive/KhoaLuan/EnTube/dataset_splits.json"
data_root = Path("/content/drive/MyDrive/KhoaLuan/EnTube/Download_2min")
batch_checkpoint_path = Path("/content/drive/MyDrive/KhoaLuan/EnTube/entube_batch_checkpoint.json")

# Mandatory keys that must exist in scene_embeddings.pt
REQUIRED_DATA_KEYS = ['embeddings', 'scene_ids', 'metadata',
                      'edge_index', 'edge_attr', 'rst_links', 'y']
OUTPUT_FIELDS = ["scene_uid", "video_id", "video_label", "caption"]

In [ ]:
with open(split_dataset_file, "r") as f:
    splits = json.load(f)
test_folders = splits["test"]

valid_folders = []
valid_folders_data = {}
invalid_folders = []

for folder_name in test_folders:
    folder_path = data_root / folder_name
    emb_path = folder_path / "scene_embeddings.pt"
    seg_path = folder_path / "segments.json"

    if not emb_path.exists() or not seg_path.exists():
        missing = []
        if not emb_path.exists():
            missing.append("scene_embeddings.pt")
        if not seg_path.exists():
            missing.append("segments.json")
        invalid_folders.append((folder_name, f"Missing file(s): {', '.join(missing)}"))
        continue

    try:
        data = torch.load(emb_path, map_location='cpu')
        missing_keys = [k for k in REQUIRED_DATA_KEYS if k not in data]
        if missing_keys:
            invalid_folders.append((folder_name, f"Missing keys in .pt: {missing_keys}"))
        else:
            valid_folders.append(folder_name)
            valid_folders_data[folder_name] = data
    except Exception as e:
        invalid_folders.append((folder_name, f"Corrupted .pt file: {str(e)}"))

print(f"\nValidation results: {len(valid_folders)} valid, {len(invalid_folders)} invalid.")
if invalid_folders:
    print("Invalid folders:")
    for f, reason in invalid_folders:
        print(f"  - {f}: {reason}")

### Khởi tạo LLM

#### HuggingFace Qwen/Qwen3-4B-Instruct-2507

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen3-4B-Instruct-2507"

# Load model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

#### HuggingFace google/gemma-4-E2B-it

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForCausalLM

model_name = "google/gemma-4-E2B-it"

# Load model
processor = AutoProcessor.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype="auto",
    device_map="auto"
)

In [ ]:
def generate_input_video_context(folder_name, data, data_root):
    """Tạo chuỗi mô tả video từ segments.json (hỗ trợ cả list và dict)."""
    seg_path = data_root / folder_name / "segments.json"
    captions_dict = {}

    try:
        with open(seg_path, 'r', encoding='utf-8') as f:
            segments_content = json.load(f)

        scene_ids_list = data['scene_ids']
        if isinstance(segments_content, list):
            for idx, seg in enumerate(segments_content):
                if idx >= len(scene_ids_list):
                    break
                s_id = int(scene_ids_list[idx])
                caption = seg.get('caption', "No text description available.")
                captions_dict[s_id] = caption
        elif isinstance(segments_content, dict):
            for k, v in segments_content.items():
                if isinstance(v, dict):
                    caption = v.get('caption', "No text description available.")
                    captions_dict[int(k)] = caption
                else:
                    captions_dict[int(k)] = str(v)
    except Exception as e:
        print(f"  [WARNING] Cannot parse segments.json of {folder_name}: {e}")

    video_context_text = ""
    video_context_text += f"- Total Extracted Scenes: {len(scene_ids_list)}\n"
    video_context_text += "- Detailed Scene Textual Descriptions:\n"

    for scene_id in scene_ids_list:
        s_id_int = int(scene_id)
        caption = captions_dict.get(s_id_int, "No caption text registered for this scene index.")
        video_context_text += f"  * Scene ID {s_id_int}: \"{caption}\"\n"

    sample_rst_str = ", ".join([str(link) for link in data['rst_links']])
    video_context_text += f"- Ground Rhetorical Structure Theory (RST) Links: [{sample_rst_str}]\n"

    return video_context_text

In [ ]:
def add_hits_to_pool(search_results, query_scene_source):
    """Gom kết quả tìm kiếm Milvus vào pool chung."""
    for hits in search_results:
        for hit in hits:
            entity = hit['entity']
            uid = entity.get('scene_uid') or hit.get('id')
            score = hit['distance']

            if uid not in aggregated_hits or score > aggregated_hits[uid]['score']:
                aggregated_hits[uid] = {
                    'score': score,
                    'scene_uid': uid,
                    'video_id': entity.get('video_id', 'N/A'),
                    'video_label': entity.get('video_label', 'N/A'),
                    'caption': entity.get('caption', 'No caption available'),
                    'matched_by_query': query_scene_source
                }

In [ ]:
def reconstruct_multimodal_subgraph_with_logs(tx, video_id, data, data_root):
    """Nạp đồ thị video test vào Neo4j."""
    print(f"=== START RECONSTRUCTING MULTIMODAL GRAPH FOR VIDEO: {video_id} ===")

    seg_path = data_root / video_id / "segments.json"
    segments_list = []

    if seg_path.exists():
        try:
            with open(seg_path, 'r', encoding='utf-8') as f:
                segments_list = json.load(f)
            print(f"--> Loaded {len(segments_list)} segments from JSON.")
        except Exception as e:
            print(f"--> [WARNING] Failed to load segments.json: {e}")
    else:
        print(f"--> [WARNING] segments.json not found at {seg_path}.")

    # Tạo node Video
    tx.run("MERGE (v:Video {id: $video_id}) SET v.is_test = true, v.updated_at = timestamp()",
           video_id=video_id)

    # Tạo các Scene node
    scene_query = """
    MATCH (v:Video {id: $video_id})
    MERGE (s:Scene {uid: $scene_uid})
    SET s.scene_id = $scene_id,
        s.caption = $caption,
        s.embedding = $embedding
    MERGE (v)-[:HAS_SCENE]->(s)
    """
    embeddings_list = data['embeddings'].tolist()
    scene_ids_list = data['scene_ids']

    for idx, scene_id in enumerate(scene_ids_list):
        scene_id_int = int(scene_id)
        scene_uid = f"{video_id}_scene_{scene_id_int}"
        caption_val = "No caption available"

        if idx < len(segments_list):
            seg_item = segments_list[idx]
            caption_val = seg_item.get('caption', caption_val)

        tx.run(scene_query,
               video_id=video_id,
               scene_uid=scene_uid,
               scene_id=scene_id_int,
               caption=caption_val,
               embedding=embeddings_list[idx])

    print(f"--> Uploaded {len(scene_ids_list)} Scene nodes.")

    # Tạo quan hệ RST
    success_rel = 0
    for src, tgt, rel_type in data.get('rst_links', []):
        src_uid = f"{video_id}_scene_{int(src)}"
        tgt_uid = f"{video_id}_scene_{int(tgt)}"

        rel_type_upper = str(rel_type).strip().upper().replace(" ", "_").replace("-", "_")

        link_query = f"""
        MATCH (src:Scene {{uid: $src_uid}}), (tgt:Scene {{uid: $tgt_uid}})
        MERGE (src)-[:{rel_type_upper}]->(tgt)
        """
        tx.run(link_query, src_uid=src_uid, tgt_uid=tgt_uid)
        success_rel += 1

    print(f"--> Connected {success_rel} RST relationships.")
    print(f"=== COMPLETED MULTIMODAL SUBGRAPH ===\n")

In [ ]:
def get_global_sequence_structural_matched_subgraphs(tx, test_video_id):
    """Truy vấn tìm đồ thị con cấu trúc tương tự."""
    query = """
    MATCH (v_test:Video {id: $test_video_id})
    MATCH p = (tsStart:Scene)-[:SIMILAR_TO|TEMPORAL|ELABORATION|CONTRAST|SPAN|ROOT|JOINT|CAUSE|TOPIC_COMMENT*2..4]->(tsEnd:Scene)
    WHERE all(s IN nodes(p) WHERE (v_test)-[:HAS_SCENE]->(s))

    MATCH (v_cand:Video)
    WHERE v_cand.id <> $test_video_id
      AND v_cand.is_test IS NULL
      AND (v_cand.video_label IS NOT NULL OR v_cand.predicted_label IS NOT NULL OR v_cand.label IS NOT NULL)

    MATCH q = (csStart:Scene)-[:SIMILAR_TO|TEMPORAL|ELABORATION|CONTRAST|SPAN|ROOT|JOINT|CAUSE|TOPIC_COMMENT*2..4]->(csEnd:Scene)
    WHERE all(s IN nodes(q) WHERE (v_cand)-[:HAS_SCENE]->(s))
      AND length(p) = length(q)
      AND all(i IN range(0, length(p)-1) WHERE type(relationships(p)[i]) = type(relationships(q)[i]))

    RETURN v_cand.id AS video_id,
           coalesce(v_cand.video_label, v_cand.predicted_label, v_cand.label) AS label,
           max(length(p) + 1) AS max_nodes_matched,
           count(distinct q) AS total_matched_sequences,
           collect(distinct {
               length: length(p) + 1,
               sequence_ids: [s IN nodes(q) | s.scene_id],
               relation_chain: [r IN relationships(q) | type(r)],
               scene_captions: [s IN nodes(q) | coalesce(s.caption, "No caption")]
           }) AS match_details
    ORDER BY max_nodes_matched DESC, total_matched_sequences DESC
    LIMIT 5
    """
    result = tx.run(query, test_video_id=test_video_id)
    return [record for record in result]

In [ ]:
import re

def extract_and_parse_json(raw_text):
    """Parser JSON + regex dự phòng."""
    if not raw_text:
        return {"predicted_label": -1, "explanation": "Empty LLM response"}
    clean_text = raw_text.strip()
    clean_text = re.sub(r'^```json\s*', '', clean_text, flags=re.IGNORECASE)
    clean_text = re.sub(r'^```\s*', '', clean_text)
    clean_text = re.sub(r'\s*```$', '', clean_text)
    clean_text = clean_text.strip()

    try:
        match = re.search(r'\{.*\}', clean_text, re.DOTALL)
        target = match.group(0) if match else clean_text
        parsed = json.loads(target)
        pred = parsed.get("predicted_label")
        if pred is not None:
            pred_str = str(pred).strip().lower()
            if any(x in pred_str for x in ['1', 'true', 'positive', 'yes']):
                parsed["predicted_label"] = 1
                return parsed
            elif any(x in pred_str for x in ['0', 'false', 'negative', 'no']):
                parsed["predicted_label"] = 0
                return parsed
    except:
        pass

    # Regex rộng
    broad = re.search(r'(predicted[-_ ]label|label|prediction)\s*["\']*?\s*[:=]\s*["\']*?\s*([a-z0-9_-]+)', clean_text, re.I)
    if broad:
        val = broad.group(2).lower()
        if any(x in val for x in ['1', 'true', 'positive', 'yes']):
            return {"predicted_label": 1, "explanation": "Broad Regex"}
        elif any(x in val for x in ['0', 'false', 'negative', 'no']):
            return {"predicted_label": 0, "explanation": "Broad Regex"}

    # Quét văn bản thô
    if re.search(r'predicted[-_ ]label\s+(is|should be)\s+1', clean_text, re.I):
        return {"predicted_label": 1, "explanation": "Raw text scan"}
    if re.search(r'predicted[-_ ]label\s+(is|should be)\s+0', clean_text, re.I):
        return {"predicted_label": 0, "explanation": "Raw text scan"}

    return {"predicted_label": -1, "explanation": "Parsing failed"}

def extract_gemma_response(parsed_response) -> str:
    """
    Normalize output of processor.parse_response() into a plain string.
    """
    if parsed_response is None:
        return ''
    if isinstance(parsed_response, dict):
        content = parsed_response.get('content') or parsed_response.get('text') or ''
        return content.strip()
    if isinstance(parsed_response, str):
        return parsed_response.strip()
    return str(parsed_response).strip()

### Chạy inference với LLM

In [ ]:
import time
import gc


system_prompt = """You are an Advanced Video Analysis and Evaluation System (VideoRAG Reasoning Engine).
Your task is to receive the structural, textual, and multimodal information of an input video, cross-reference it with the similar contexts retrieved from the Vector Database and the Knowledge Graph experience, and provide the final judgment.
"""

# Khôi phục checkpoint nếu có
evaluation_results = []
processed_folders = set()
if batch_checkpoint_path.exists():
    try:
        with open(batch_checkpoint_path, 'r', encoding='utf-8') as f:
            evaluation_results = json.load(f)
        processed_folders = {res['folder_name'] for res in evaluation_results}
        print(f"[INFO] Resuming from checkpoint. Processed {len(processed_folders)} videos.")
    except Exception as e:
        print(f"[WARNING] Corrupted checkpoint, starting fresh: {e}")

num_videos_to_runs = len(valid_folders)
final_queue = [f for f in valid_folders if f not in processed_folders][:num_videos_to_runs]
print(f"[START] Will process {len(final_queue)} videos.\n")

success_count = 0

for current_idx, folder in enumerate(final_queue, 1):
    print(f"\n{'='*70}")
    print(f" $$ PROCESSING VIDEO [{current_idx}/{len(final_queue)}]: {folder}")
    print("="*70)

    test_video_id = folder
    # Lấy data đã load sẵn
    sample_data = valid_folders_data[folder]

    # Khởi tạo pool kết quả Milvus cho video này
    aggregated_hits = {}
    milvus_context_text = ""
    neo4j_context_text = ""

    try:
        # BƯỚC A: Tạo ngữ cảnh văn bản
        video_context_text = generate_input_video_context(folder, sample_data, data_root)

        # BƯỚC B: Truy vấn Milvus
        scene_embeddings = sample_data['embeddings']
        num_scenes = scene_embeddings.shape[0]

        for i in range(num_scenes):
            res_scene = milvus_client.search(
                collection_name=COLLECTION_NAME,
                data=[scene_embeddings[i].tolist()],
                limit=5,
                output_fields=OUTPUT_FIELDS
            )
            add_hits_to_pool(res_scene, query_scene_source=f"Query Scene {sample_data['scene_ids'][i]}")

        res_mean = milvus_client.search(
            collection_name=COLLECTION_NAME,
            data=[scene_embeddings.mean(dim=0).tolist()],
            limit=5,
            output_fields=OUTPUT_FIELDS
        )
        add_hits_to_pool(res_mean, query_scene_source="Video-level Mean Vector")

        top_5_hits = sorted(aggregated_hits.values(), key=lambda x: x['score'], reverse=True)[:5]
        count_label_1, count_label_0 = 0, 0
        detailed_matches_text = ""

        for idx, hit in enumerate(top_5_hits, 1):
            if hit['video_label'] == 1: count_label_1 += 1
            elif hit['video_label'] == 0: count_label_0 += 1

            s_uid = hit['scene_uid']
            retrieved_scene_idx = s_uid.split('_')[-1] if '_' in str(s_uid) else "Unknown"
            detailed_matches_text += (
                f"- Top {idx} Match (Cosine Score: {hit['score']:.4f}):\n"
                f"  Matched by: {hit['matched_by_query']}\n"
                f"  Target Location: Found Video ID '{hit['video_id']}' at Scene index '{retrieved_scene_idx}'\n"
                f"  Target Video Ground-Truth Label: {hit['video_label']}\n"
                f"  Scene Caption: {hit['caption']}\n\n"
            )

        milvus_context_text = (
            "=== MILVUS VECTOR RETRIEVAL SUMMARY ===\n"
            f"Among the top 5 most similar retrieved video segments, there are {count_label_1} instances with Label 1 and {count_label_0} instances with Label 0.\n\n"
            "=== DETAILED RETRIEVED SEGMENTS ===\n" + detailed_matches_text
        )

        # BƯỚC C: Neo4j - nạp đồ thị và truy vấn
        with driver.session(database=DATABASE) as session:
            session.execute_write(reconstruct_multimodal_subgraph_with_logs, test_video_id, sample_data, data_root)

        with driver.session(database=DATABASE) as session:
            records = session.execute_read(get_global_sequence_structural_matched_subgraphs, test_video_id)
            if records:
                for idx, record in enumerate(records, 1):
                    neo4j_context_text += f"- Top {idx} Structural Match: Video '{record['video_id']}' (Label: {record['label']})\n"
                    neo4j_context_text += f"  Max nodes matched: {record['max_nodes_matched']}, Total paths: {record['total_matched_sequences']}\n"
                    for path_idx, det in enumerate(record['match_details'][:2], 1):
                        caps = ' | '.join(det['scene_captions'])
                        neo4j_context_text += (
                            f"  + Path Sample {path_idx}:\n"
                            f"    Scene Sequence: {det['sequence_ids']}\n"
                            f"    RST Logic Chain: [{' -> '.join(det['relation_chain'])}]\n"
                            f"    Semantic Flow: {caps}\n"
                        )
                    neo4j_context_text += "\n"
            else:
                neo4j_context_text = "No relevant structural context found in Knowledge Graph.\n"

        # (Sau khi dùng xong Neo4j, có thể xóa đồ thị test ngay để giảm tải)
        with driver.session(database=DATABASE) as session:
            session.run("MATCH (v:Video {id: $id}) DETACH DELETE v", id=test_video_id)
            print(f"  [CLEANUP] Removed test graph for {test_video_id}.")

        # BƯỚC D: Xây dựng prompt
        llm_prompt = f"""=== 1. CURRENT INPUT VIDEO STRUCTURE & CONTENT ===
{video_context_text}

=== 2. CORE SEMANTIC SIMILAR CONTEXT (From Milvus Vector DB) ===
{milvus_context_text}

=== 3. RELEVANT LOGICAL STRUCTURAL CONTEXT (From Neo4j Knowledge Graph Subgraph) ===
{neo4j_context_text}

=== OUTPUT COMPLIANCE ===
Analyze the content descriptions, visual/audio elements, and the spatio-temporal structural characteristics (RST links). Provide your final accurate judgment in the following strict JSON format:
{{
  "predicted_label": "appropriate_predicted_label",
  "explanation": "Detailed multimodal analysis clarifying the similarity in content flow, visual/audio elements, and rst_links logic with the database context provided above.",
  "improvement_suggestions": [
    "Actionable suggestion to optimize the logical narrative flow or video production quality 1",
    "Actionable suggestion to optimize the logical narrative flow or video production quality 2"
  ]
}}
"""
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": llm_prompt},
        ]

        # # 1. Qwen/Qwen3-4B-Instruct-2507
        # text = tokenizer.apply_chat_template(
        #     messages,
        #     tokenize=False,
        #     add_generation_prompt=True
        # )

        # # Kiểm tra độ dài token, nếu quá dài có thể cắt bớt
        # if len(tokenizer.encode(text)) > 32768:
        #     print(f"  [WARNING] Prompt too long ({len(tokenizer.encode(text))}), truncating Neo4j context.")
        #     llm_prompt = llm_prompt.replace(neo4j_context_text, "Context truncated due to size limitations.\n")
        #     messages = [
        #         {"role": "system", "content": system_prompt},
        #         {"role": "user", "content": llm_prompt},
        #     ]

        #     text = tokenizer.apply_chat_template(
        #         messages,
        #         tokenize=False,
        #         add_generation_prompt=True
        #     )

        # # Inference
        # model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
        # generated_ids = model.generate(
        #     **model_inputs,
        #     max_new_tokens=1024
        # )
        # output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()
        # llm_response = tokenizer.decode(output_ids, skip_special_tokens=True).strip()

        # 2. google/gemma-4-E2B-it
        text = processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False
        )

        model_inputs = processor(text=text, return_tensors="pt").to(model.device)
        input_len = model_inputs["input_ids"].shape[-1]

        # Generate output
        outputs = model.generate(**model_inputs, max_new_tokens=1024)
        response = processor.decode(outputs[0][input_len:], skip_special_tokens=False)

        # Parse output
        parsed_response = processor.parse_response(response)
        llm_response = extract_gemma_response(parsed_response)

        # BƯỚC F: Xử lý kết quả
        ground_truth_label = sample_data['y'].item() if isinstance(sample_data['y'], torch.Tensor) else sample_data['y']
        verdict = extract_and_parse_json(llm_response)
        pred_label = verdict.get('predicted_label', -1)

        record = {
            "folder_name": folder,
            "db_test_id": test_video_id,
            "ground_truth": ground_truth_label,
            "prediction": pred_label,
            "explanation": verdict.get('explanation', 'Parsing failed'),
            "raw_llm_output": llm_response
        }
        evaluation_results.append(record)

        status = "[MATCHED]" if ground_truth_label == pred_label else ("[MISMATCHED]" if pred_label != -1 else "[PARSE_ERROR]")
        print(f"  --> {status} | GT: {ground_truth_label}, Pred: {pred_label}")

        success_count += 1

        # Lưu checkpoint
        with open(batch_checkpoint_path, 'w', encoding='utf-8') as f:
            json.dump(evaluation_results, f, ensure_ascii=False, indent=4)

    except Exception as e:
        print(f"  [CRITICAL ERROR] {folder}: {e}")
        # Vẫn cố gắng dọn dẹp đồ thị test nếu đã được nạp
        try:
            with driver.session(database=DATABASE) as session:
                session.run("MATCH (v:Video {id: $id}) DETACH DELETE v", id=test_video_id)
        except:
            pass
        continue

    finally:
        # Dọn dẹp bộ nhớ GPU
        if 'model_inputs' in locals(): del model_inputs
        if 'outputs' in locals(): del outputs
        # if 'generated_ids' in locals(): del generated_ids
        # if 'output_ids' in locals(): del output_ids
        gc.collect()
        torch.cuda.empty_cache()
        time.sleep(0.4)


print("\n" + "="*70)
print(f"$ PIPELINE COMPLETED. Processed {success_count} videos.")
print(f"Results saved to {batch_checkpoint_path}")
print("="*70)

# Đóng kết nối Neo4j
try:
    driver.close()
    print("[INFO] Neo4j connection closed.")
except:
    pass

### Metrics

In [ ]:
import os
import json
from pathlib import Path

batch_checkpoint_path = Path("/content/drive/MyDrive/KhoaLuan/EnTube_Small/entube_test_100_batch_checkpoint.json")

if not os.path.exists(batch_checkpoint_path):
    print(f"[ERROR] Khong tim thay file checkpoint kết quả tại: {batch_checkpoint_path}")
    print("Vui lòng đảm bảo tiến trình chạy batch ở Cell trước đã hoàn thành và lưu dữ liệu.")
else:
    # 1. Đọc tệp dữ liệu kết quả đóng băng từ Drive
    with open(batch_checkpoint_path, 'r', encoding='utf-8') as f:
        results_data = json.load(f)

    total_samples = len(results_data)
    correct_predictions = 0
    parsing_errors = 0

    # Khởi tạo ma trận nhầm lẫn (Confusion Matrix)
    # Cấu trúc: matrix[ground_truth][prediction]
    # Lớp nhãn hợp lệ gồm 0 và 1
    tp, fp, fn, tn = 0, 0, 0, 0

    print("=" * 75)
    print(f"$$ SYSTEM PERFORMANCE REPORT - TOTAL VIDEOS EVALUATED: {total_samples}")
    print("=" * 75)
    print(f"{'No.':<5} | {'Folder Name':<35} | {'GT':<4} | {'PRED':<4} | {'Status':<12}")
    print("-" * 75)

    for idx, res in enumerate(results_data, 1):
        gt = res['ground_truth']
        pred = res['prediction']
        folder_name = res['folder_name']

        # Chuẩn hóa định dạng nhãn dự đoán phòng hờ kiểu ký tự lạ
        try:
            pred_int = int(float(pred)) if str(pred).replace('.','',1).isdigit() or isinstance(pred, (int, float)) else -1
        except:
            pred_int = -1

        # Kiểm tra trạng thái khớp nhãn
        if pred_int == -1:
            status = "PARSE_ERROR"
            parsing_errors += 1
        elif gt == pred_int:
            status = "MATCHED"
            correct_predictions += 1
            if gt == 1: tp += 1  # True Positive
            else: tn += 1        # True Negative
        else:
            status = "MISMATCHED"
            if gt == 1 and pred_int == 0: fn += 1  # False Negative
            if gt == 0 and pred_int == 1: fp += 1  # False Positive

        # In log chi tiết rút gọn cho từng hàng mục
        short_folder = folder_name if len(folder_name) <= 33 else folder_name[:30] + "..."
        print(f"{idx:03d}   | {short_folder:<35} | {gt:<4} | {pred_int:<4} | {status:<12}")

    # 2. Tính toán các chỉ số thống kê nâng cao
    valid_evaluated_samples = total_samples - parsing_errors

    accuracy = (correct_predictions / total_samples) * 100 if total_samples > 0 else 0

    # Chỉ số riêng cho Lớp Nhãn 1 (Positive)
    precision_1 = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall_1 = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1_1 = 2 * (precision_1 * recall_1) / (precision_1 + recall_1) if (precision_1 + recall_1) > 0 else 0

    # Chỉ số riêng cho Lớp Nhãn 0 (Negative)
    precision_0 = tn / (tn + fn) if (tn + fn) > 0 else 0
    recall_0 = tn / (tn + fp) if (tn + fp) > 0 else 0
    f1_0 = 2 * (precision_0 * recall_0) / (precision_0 + recall_0) if (precision_0 + recall_0) > 0 else 0

    # 3. In bảng tổng kết hiệu năng đồ thị và mô hình RAG
    print("\n" + "=" * 75)
    print(" FINAL PIPELINE PERFORMANCE METRICS")
    print("=" * 75)
    print(f"• Total Dataset Processed:           {total_samples:,} videos")
    print(f"• Total Correct Predictions (Hits):  {correct_predictions:,} videos")
    print(f"• Total Failed Structural Parses:    {parsing_errors:,} videos")
    print(f"• OVERALL ACCURACY RATE:             {accuracy:.2f}%")
    print("-" * 75)
    print(" DETAILED CONFUSION MATRIX:")
    print(f"  - True Positive (GT=1, Pred=1):   {tp}")
    print(f"  - False Positive (GT=0, Pred=1):  {fp}")
    print(f"  - False Negative (GT=1, Pred=0):  {fn}")
    print(f"  - True Negative (GT=0, Pred=0):   {tn}")
    print("-" * 75)
    print(" CLASS-LEVEL ANALYSIS:")
    print(f"  [LABEL 1] Precision: {precision_1*100:.2f}% | Recall: {recall_1*100:.2f}% | F1-Score: {f1_1*100:.2f}%")
    print(f"  [LABEL 0] Precision: {precision_0*100:.2f}% | Recall: {recall_0*100:.2f}% | F1-Score: {f1_0*100:.2f}%")
    print("=" * 75)